# Feedforward classification of the NMIST data
### Advanced Deep Learning 2024
This notebook was written originally Jon Sporring (mailto:sporring@di.ku.dk) and heavily inspired by https://clay-atlas.com/us/blog/2021/04/22/pytorch-en-tutorial-4-train-a-model-to-classify-mnist.

We consider the Modified National Institute of Standards and Technology database of handwritten digits (MNIST): http://yann.lecun.com/exdb/mnist/

## Installs

On non-colab system, is usually good to make an environment and install necessary tools there. E.g., anaconda->jupyter->terminal create an environment, if you have not already, and activate it:
```
conda create -n adl python=3.9
conda activate adl
```
then install missing packages such as:
```
conda install ipykernel torch matplotlib torchmetrics scikit-image jpeg
conda install -c conda-forge segmentation-models-pytorch ipywidgets
```
and if you want to add it to jupyter's drop-down menu
```
ipython kernel install --user --name=adl
```
Now reload the jupyter-notebook's homepage and make a new or load an existing file. On colab, the tools have to be installed everytime

In [ ]:
'''try:
  import google.colab
  IN_COLAB = True
except:
  IN_COLAB = False
if IN_COLAB:
    !pip3 install torch matplotlib torchmetrics scikit-image segmentation-models-pytorch'''

## Imports

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as dset
from torchvision import datasets, transforms

## Set global device

In [2]:
# GPU
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print('GPU State:', device)

GPU State: cpu


## Functions

In [3]:
def training_loop(model, loss, optimizer, loader, epochs, verbose=True, device=device):
    """
    Run training of a model given a loss function, optimizer and a set of training and validation data.
    """

    # Train
    for epoch in range(epochs):
        running_loss = 0.0

        for times, data in enumerate(loader):
            inputs, labels = data[0].to(device), data[1].to(device)
            inputs = inputs.view(inputs.shape[0], -1)

            # Zero the parameter gradients
            optimizer.zero_grad()

            # Foward + backward + optimize
            outputs = model(inputs)
            loss_tensor = loss(outputs, labels)
            loss_tensor.backward()
            optimizer.step()

            # Print statistics
            running_loss += loss_tensor.item()
            if verbose:
                if times % 100 == 99 or times+1 == len(loader):
                    print('[%d/%d, %d/%d] loss: %.3f' % (epoch+1, epochs, times+1, len(loader), running_loss/2000))

In [4]:
def evaluate_model(model, loader, device=device):
    """
    Evaluate a model 'model' on all batches of a torch DataLoader 'data_loader'.

    Returns: the total number of correct classifications,
             the total number of images
             the list of the per class correct classification,
             the list of the per class total number of images.
    """

    # Test
    correct = 0
    total = 0

    with torch.no_grad():
        for data in loader:
            inputs, labels = data
            inputs, labels = inputs.to(device), labels.to(device)
            inputs = inputs.view(inputs.shape[0], -1)

            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    class_correct = [0 for i in range(10)]
    class_total = [0 for i in range(10)]

    with torch.no_grad():
        for data in loader:
            inputs, labels = data[0].to(device), data[1].to(device)
            inputs = inputs.view(inputs.shape[0], -1)

            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            c = (predicted == labels).squeeze()
            for i in range(len(labels)):
                label = labels[i]
                class_correct[label] += c[i].item()
                class_total[label] += 1

    return (correct, total, class_correct, class_total)


## Main program

In [5]:
# Transform
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5,), (0.5,)),]
)

In [6]:
# Data
trainSet = datasets.MNIST(root='MNIST', download=True, train=True, transform=transform)
testSet = datasets.MNIST(root='MNIST', download=True, train=False, transform=transform)
trainLoader = dset.DataLoader(trainSet, batch_size=64, shuffle=True)
testLoader = dset.DataLoader(testSet, batch_size=64, shuffle=False)

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 9912422/9912422 [00:01<00:00, 5124391.00it/s]


Extracting MNIST/MNIST/raw/train-images-idx3-ubyte.gz to MNIST/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 28881/28881 [00:00<00:00, 247383.32it/s]


Extracting MNIST/MNIST/raw/train-labels-idx1-ubyte.gz to MNIST/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 1648877/1648877 [00:00<00:00, 2068579.26it/s]


Extracting MNIST/MNIST/raw/t10k-images-idx3-ubyte.gz to MNIST/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 4542/4542 [00:00<00:00, 4802250.76it/s]

Extracting MNIST/MNIST/raw/t10k-labels-idx1-ubyte.gz to MNIST/MNIST/raw



In [7]:
# Model
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.main = nn.Sequential(
            nn.Linear(in_features=784, out_features=128),
            nn.ReLU(),
            nn.Linear(in_features=128, out_features=64),
            nn.ReLU(),
            nn.Linear(in_features=64, out_features=10),
            nn.LogSoftmax(dim=1)
        )

    def forward(self, input):
        return self.main(input)


net = Net().to(device)
print(net)

Net(
  (main): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
    (5): LogSoftmax(dim=1)
  )
)


In [8]:
# Parameters
epochs = 4
lr = 0.002
loss = nn.NLLLoss()
optimizer = optim.SGD(net.parameters(), lr=0.002, momentum=0.9)

# Train
print('Training on %d images' % trainSet.data.shape[0])
training_loop(net, loss, optimizer, trainLoader, epochs)
print('Training Finished.\n')

# Test
correct, total, class_correct, class_total = evaluate_model(net, testLoader)
print('Accuracy of the network on the %d test images: %d %%' % (testSet.data.shape[0], (100*correct / total)))
for i in range(10):
    print('Accuracy of %d: %3f' % (i, (class_correct[i]/class_total[i])))

Training on 60000 images
[1/4, 100/938] loss: 0.107
[1/4, 200/938] loss: 0.176
[1/4, 300/938] loss: 0.214
[1/4, 400/938] loss: 0.242
[1/4, 500/938] loss: 0.265
[1/4, 600/938] loss: 0.287
[1/4, 700/938] loss: 0.308
[1/4, 800/938] loss: 0.327
[1/4, 900/938] loss: 0.344
[1/4, 938/938] loss: 0.352
[2/4, 100/938] loss: 0.018
[2/4, 200/938] loss: 0.033
[2/4, 300/938] loss: 0.049
[2/4, 400/938] loss: 0.065
[2/4, 500/938] loss: 0.081
[2/4, 600/938] loss: 0.097
[2/4, 700/938] loss: 0.112
[2/4, 800/938] loss: 0.127
[2/4, 900/938] loss: 0.142
[2/4, 938/938] loss: 0.147
[3/4, 100/938] loss: 0.015
[3/4, 200/938] loss: 0.029
[3/4, 300/938] loss: 0.043
[3/4, 400/938] loss: 0.056
[3/4, 500/938] loss: 0.069
[3/4, 600/938] loss: 0.081
[3/4, 700/938] loss: 0.093
[3/4, 800/938] loss: 0.105
[3/4, 900/938] loss: 0.117
[3/4, 938/938] loss: 0.121
[4/4, 100/938] loss: 0.012
[4/4, 200/938] loss: 0.023
[4/4, 300/938] loss: 0.034
[4/4, 400/938] loss: 0.045
[4/4, 500/938] loss: 0.056
[4/4, 600/938] loss: 0.067
[4/

In [9]:
# Count parameters
total_params = sum(p.numel() for p in net.parameters())
trainable_params = sum(p.numel() for p in net.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Detailed breakdown
print("\nParameter breakdown by layer:")
for name, param in net.named_parameters():
    print(f"{name}: {param.numel():,} parameters (shape: {list(param.shape)})")

Total parameters: 109,386
Trainable parameters: 109,386

Parameter breakdown by layer:
main.0.weight: 100,352 parameters (shape: [128, 784])
main.0.bias: 128 parameters (shape: [128])
main.2.weight: 8,192 parameters (shape: [64, 128])
main.2.bias: 64 parameters (shape: [64])
main.4.weight: 640 parameters (shape: [10, 64])
main.4.bias: 10 parameters (shape: [10])


## **Answers to Task 1.1**



### **1. Brief Description of Each Part of the Program**

**Imports Section**: Imports necessary PyTorch libraries including neural network modules (`nn`), optimization (`optim`), data utilities (`dset`), and vision datasets/transforms.

**Device Configuration**: Sets up the computation device (CPU or GPU) for training and inference.

### Functions Section:

`training_loop()`: Implements the training procedure that iterates through epochs and batches, performs forward pass, computes loss, backpropagates gradients, and updates model parameters using the optimizer.

`evaluate_model()`: Evaluates the trained model on a dataset, computing overall accuracy and per-class accuracy for all 10 digit classes.

### Main Program:

**Transform**: Defines data preprocessing pipeline to convert images to tensors and normalize pixel values.

**Data Loading**: Downloads and loads MNIST training and test datasets, creates DataLoaders with batch size 64.

**Model Definition**: Creates a feedforward neural network (`Net`) with three fully connected layers (784→128→64→10) with ReLU activations and LogSoftmax output.

**Training**: Trains the model for 4 epochs using SGD optimizer with learning rate 0.002 and momentum 0.9, with NLLLoss as the loss function.

**Evaluation**: Tests the model on the test set and reports overall and per-class accuracies.

### **2. What does the `transform` function achieve?**

The `transform` function applies two preprocessing steps to the MNIST images:

1. `transforms.ToTensor()`: Converts PIL images or numpy arrays to PyTorch tensors and scales pixel values from [0, 255] to [0.0, 1.0] by dividing by 255.

2. `transforms.Normalize((0.5,), (0.5,))`: Normalizes the tensor values using the formula: `(pixel - mean) / std`. With mean=0.5 and std=0.5, this transforms pixel values from [0.0, 1.0] to approximately [-1.0, 1.0]. This helps with training stability and convergence.

Together, these transforms prepare the raw image data (0-255 integers) into normalized tensors suitable for neural network training.

### **3. How many parameters does this model use?**

**Total parameters: 109,386**

Breakdown by layer:
- **Layer 1** (784 → 128): 100,352 weights + 128 biases = **100,480 parameters**
- **Layer 2** (128 → 64): 8,192 weights + 64 biases = **8,256 parameters**
- **Layer 3** (64 → 10): 640 weights + 10 biases = **650 parameters**

All parameters are trainable.

### **4. How well is it able to correctly classify each digit class 0, 1, ..., 9?**

**Overall Test Accuracy: 94%** (on 10,000 test images)

**Per-class accuracy:**
- **Digit 0**: 98.16% (highest accuracy)
- **Digit 1**: 98.15% (highest accuracy)
- **Digit 6**: 95.20%
- **Digit 7**: 94.84%
- **Digit 9**: 94.45%
- **Digit 4**: 93.89%
- **Digit 3**: 93.76%
- **Digit 2**: 93.51%
- **Digit 8**: 91.17%
- **Digit 5**: 90.13% (lowest accuracy)

The model performs best on digits 0 and 1 (both ~98.2% accuracy). Digit 5 has the lowest accuracy at 90.13%, suggesting it may be confused with similar digits (possibly 3, 6, or 8). All digit classes achieve at least 90% accuracy, indicating the model generalizes reasonably well across all classes.

## **Answers to Task 1.2**

Current loss function: The model uses `nn.NLLLoss()` (Negative Log Likelihood Loss). This is appropriate because the model outputs LogSoftmax values (line 5 in the network). Loss function with logits: If we remove the LogSoftmax layer (line 5) and use linear outputs (logits), we should use `nn.CrossEntropyLoss()` instead. Doing this combines `LogSoftmax` + `NLLLoss` in a single, numerically stable operation. Using `CrossEntropyLoss` with logits is more efficient and numerically stable than applying softmax/log-softmax separately

Softmax can cause overflow/underflow with large logit values. Whereas, log-softmax uses the log-sum-exp trick for numerical stability. `CrossEntropyLoss` with logits computes softmax in log-space, avoiding intermediate probability values. Furthermore, softmax probabilities can saturate (values near 0 or 1), leading to vanishing gradients, where on the other hand working with logits/log-probabilities provides better gradient signals during backpropagation.

In summary, using logits with `CrossEntropyLoss` is the recommended approach because it is numerically stable, computationally efficient, and provides better gradient flow compared to explicitly computing softmax probabilities.